In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# 1. 设定站点文件夹路径
data_dir = r"C:\Users\11823\Documents\Claude\Projects\Greenlandalbedo\StationData\day"

# 2. 获取所有 csv 文件
csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
print(f"共找到 {len(csv_files)} 个站点文件。\n")

summary_list = []

# 3. 遍历检查每个站点
for file in csv_files:
    station_name = os.path.basename(file).replace(".csv", "")
    
    try:
        df = pd.read_csv(file)
        
        # 确保时间列解析正确
        if 'time' not in df.columns:
            print(f"警告: {station_name} 缺失 'time' 列，跳过！")
            continue
            
        df['time'] = pd.to_datetime(df['time'])
        df = df.sort_values('time').reset_index(drop=True)
        
        # 统计时间轴信息
        start_date = df['time'].min().strftime('%Y-%m-%d')
        end_date = df['time'].max().strftime('%Y-%m-%d')
        total_rows = len(df)
        
        # 检查时间轴是否有断天 (Missing Dates)
        full_date_range = pd.date_range(start=df['time'].min(), end=df['time'].max(), freq='D')
        missing_days_count = len(full_date_range) - total_rows
        
        # 检查重复日期
        duplicate_days = df['time'].duplicated().sum()
        
        # 关键目标变量缺失率检查
        # 太阳辐射 (以 dsr_cor 优先，没有则用 dsr)
        rad_col = 'dsr_cor' if 'dsr_cor' in df.columns else 'dsr'
        rad_missing_pct = df[rad_col].isna().mean() * 100 if rad_col in df.columns else 100.0
        
        # 风速 (wspd_u)
        wind_col = 'wspd_u'
        wind_missing_pct = df[wind_col].isna().mean() * 100 if wind_col in df.columns else 100.0
        
        # 反照率 (albedo)
        albedo_col = 'albedo'
        albedo_missing_pct = df[albedo_col].isna().mean() * 100 if albedo_col in df.columns else 100.0
        
        # 气温 (t_u)
        temp_col = 't_u'
        temp_missing_pct = df[temp_col].isna().mean() * 100 if temp_col in df.columns else 100.0
        
        # 物理异常值初筛（极地范围界定）
        # 辐射负值（排除微小噪声，找 < -10 W/m2 的异常值）
        rad_anom = (df[rad_col] < -10).sum() if rad_col in df.columns else 0
        # 风速负值或极端离群点 (> 60 m/s)
        wind_anom = ((df[wind_col] < 0) | (df[wind_col] > 60)).sum() if wind_col in df.columns else 0
        
        # 获取站点大致坐标 (取非空的第一个值)
        lat = df['lat'].dropna().iloc[0] if 'lat' in df.columns and not df['lat'].dropna().empty else np.nan
        lon = df['lon'].dropna().iloc[0] if 'lon' in df.columns and not df['lon'].dropna().empty else np.nan

        summary_list.append({
            "Station": station_name,
            "Lat": lat,
            "Lon": lon,
            "Start_Date": start_date,
            "End_Date": end_date,
            "Total_Days": total_rows,
            "Missing_Dates": missing_days_count,
            "Duplicate_Dates": duplicate_days,
            "Rad_Missing_%": round(rad_missing_pct, 1),
            "Wind_Missing_%": round(wind_missing_pct, 1),
            "Albedo_Missing_%": round(albedo_missing_pct, 1),
            "Temp_Missing_%": round(temp_missing_pct, 1),
            "Rad_Anom_Cnt": rad_anom,
            "Wind_Anom_Cnt": wind_anom
        })
        
    except Exception as e:
        print(f"读取文件 {station_name} 失败: {str(e)}")

# 4. 转为 DataFrame 展示诊断报告
audit_df = pd.DataFrame(summary_list)

# 按辐射缺失率升序排列，方便快速筛选可用性高的站点
audit_df = audit_df.sort_values("Rad_Missing_%").reset_index(drop=True)

# 打印全面统计结果
print("=== 站点数据完整性诊断摘要报告 ===")
display(audit_df.head(20)) # 在 ipynb 中优雅地预览前 20 个站点

# 5. 保存诊断结果备用
# 修改为保存到当前脚本所在目录（或者项目根目录）
output_report_path = "station_data_audit_report.csv"
audit_df.to_csv(output_report_path, index=False, encoding='utf-8-sig')
print(f"完整诊断报告已成功保存至当前目录: {os.path.abspath(output_report_path)}")

共找到 53 个站点文件。

=== 站点数据完整性诊断摘要报告 ===


,Station,Lat,Lon,Start_Date,End_Date,Total_Days,Missing_Dates,Duplicate_Dates,Rad_Missing_%,Wind_Missing_%,Albedo_Missing_%,Temp_Missing_%,Rad_Anom_Cnt,Wind_Anom_Cnt
0,KAN_B_day,67.130000,-50.180000,2011-04-13,2026-07-06,5564,0,0,0.3,1.1,50.8,0.1,0,0
1,SER_B_day,65.679719,-37.917360,2024-07-14,2026-07-06,723,0,0,0.8,53.5,51.0,0.6,0,0
2,QAS_L_day,61.024202,-46.872457,2007-08-24,2026-07-06,6786,106,0,1.2,12.6,47.6,0.4,0,0
3,NUK_N_day,64.945811,-49.883677,2010-07-25,2014-07-25,1380,82,0,2.5,23.8,54.3,0.8,0,0
4,FRE_day,74.387584,-20.832251,2016-05-07,2026-07-06,3712,1,0,4.6,2.5,66.8,0.8,0,0
5,STA_day,70.650921,-35.823815,2026-06-12,2026-07-06,25,0,0,8.0,8.0,4.0,8.0,0,0
6,SUM_day,72.595771,-38.473583,2026-06-14,2026-07-06,23,0,0,8.7,8.7,4.3,8.7,0,0
7,QAS_A_day,61.245373,-46.733793,2012-08-20,2015-08-24,580,520,0,9.8,8.4,59.3,7.4,0,0
8,SCO_U_day,72.394385,-27.264289,2008-07-21,2026-07-06,6115,445,0,10.5,0.7,59.1,4.8,0,0
9,KAN_L_day,67.097322,-49.931820,2008-09-01,2026-07-06,6518,0,0,11.9,0.5,52.0,0.1,0,0


完整诊断报告已成功保存至当前目录: c:\Users\11823\era5land\station_data_audit_report.csv
